# Instella Reasoning — GPU Experiment Suite

Two cells. **Cell 1** does everything: setup (idempotent — safe to re-run any time) and the
suite at the `LIMIT` set at the top. Run it once with `LIMIT = 20` (sanity, ~45-60 min),
check the accuracy table it prints, then set `LIMIT = 200` and run it again (~5-10 h).
**Cell 2** zips the results for download.

Prereqs: Runtime → T4 GPU · Secrets (🔑): `github` (GitHub PAT, repo read) and `HF_TOKEN`
(HuggingFace read), both with Notebook access ON.

In [ ]:
LIMIT = 20   # sanity pass; set to 200 for the real run and re-run this cell

# ---- setup (idempotent: clone or pull, install only if missing) ----
import os, subprocess, glob, json
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get("github")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_DISABLE_XET"] = "1"

if not os.path.isdir("/content/Instella-Reasoning"):
    !git clone -b agent/cpu-experiment-suite https://$GITHUB_TOKEN@github.com/GIND123/Instella-Reasoning /content/Instella-Reasoning
%cd /content/Instella-Reasoning
!git pull

if subprocess.run(["which", "instella-reasoning"], capture_output=True).returncode != 0:
    !pip install -q -e ".[hf,retrieval,viz,stats]"
    !pip install -q "bitsandbytes>=0.46.1"

# ---- run the suite ----
!bash experiments/run_gpu_suite.sh {LIMIT}

# ---- accuracy quick-look ----
print(f"\n=== accuracy per (model x benchmark), LIMIT={LIMIT} ===")
for f in sorted(glob.glob("experiments/runs/*_gpu-suite/scores/*__*.jsonl")):
    if f.endswith("__ALL.jsonl"):
        continue
    rows = [json.loads(l) for l in open(f) if l.strip()]
    if rows:
        acc = sum(r["correct"] for r in rows) / len(rows)
        print(f"{f.split('/')[-1]:50} {len(rows):4} items  acc={acc:.3f}")

In [ ]:
# Zip results and download. Unzip into the repo so the folder lands at
# experiments/runs/<date>_gpu-suite/, then hand it to the analysis step.
!zip -qr gpu_results.zip experiments/runs/*_gpu-suite
from google.colab import files
files.download("gpu_results.zip")